# **Regresión Logística**

# Fase 1: Carga de dataset y arranque

**1. Carga del dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**2. Instalaciónn e importación de librerías**


In [ ]:
import pandas as pd

!pip install statsmodels plotly

**3. Exploración rápida**

In [ ]:
ruta = "/ruta donde tienes guardado el dataset"
df = pd.read_csv(ruta)

print(df.shape)              # 30000 Filas x 25 columnas
print(df.columns.tolist())
df.head()

(30000, 25)
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default.payment.next.month']


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


**4. Limpieza y guardado**

In [ ]:
df = df.rename(columns={"default.payment.next.month": "default"}) # Estandarización de column name
df = df.drop(columns=["ID"], errors="ignore") # Elimar columna ID y errors es para que si se corre la línea dos veces, no morir
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4}) # No hay documentación, así que a "otros"
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3}) # No hay documentación, así que a "otros"

print(df.isnull().sum().sum(), df.duplicated().sum()) # Total valores nulos en la tabla
print(df["default"].value_counts(normalize=True)) # Cantidad de filas repetidas

df.to_csv("/content/drive/MyDrive/Regresión Logística - Proyecto/credit_clean.csv", index=False)

0 35
default
0    0.7788
1    0.2212
Name: proportion, dtype: float64


In [ ]:
dups = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print(dups.shape)
dups

(70, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
12430,20000.0,1,2,2,24,2,2,4,4,4,...,1650.0,1650.0,1650.0,0.0,0.0,0.0,0.0,0.0,0.0,1
14294,20000.0,1,2,2,24,2,2,4,4,4,...,1650.0,1650.0,1650.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1759,50000.0,1,2,2,26,1,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
10250,50000.0,1,2,2,26,1,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
7170,50000.0,2,1,2,23,1,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27966,360000.0,2,1,2,27,1,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
18325,360000.0,2,1,2,29,1,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
22162,360000.0,2,1,2,29,1,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
839,500000.0,1,1,1,43,1,-2,-2,-2,-2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


**5. Tabla de descripción de variables**

In [ ]:
descripcion = pd.DataFrame([
    ["ID", "Identificador del cliente", "Identificador", "Entero"],
    ["LIMIT_BAL", "Monto del crédito otorgado (NTD), incluye crédito individual y familiar", "Cuantitativa continua", "NTD"],
    ["SEX", "Sexo del cliente", "Cualitativa nominal (binaria)", "1 = hombre, 2 = mujer"],
    ["EDUCATION", "Nivel de escolaridad", "Cualitativa ordinal", "1 = posgrado, 2 = universidad, 3 = bachillerato, 4 = otros"],
    ["MARRIAGE", "Estado civil", "Cualitativa nominal", "1 = casado, 2 = soltero, 3 = otros"],
    ["AGE", "Edad del cliente", "Cuantitativa discreta", "Años"],
    ["PAY_0", "Estado del pago de septiembre de 2005 (el más reciente)", "Cualitativa ordinal", "-2 = sin consumo, -1 = pago total, 0 = pago mínimo, 1 a 9 = meses de atraso"],
    ["PAY_2 a PAY_6", "Estado del pago de agosto, julio, junio, mayo y abril de 2005", "Cualitativa ordinal", "Misma escala que PAY_0"],
    ["BILL_AMT1 a BILL_AMT6", "Monto de la factura de septiembre a abril de 2005", "Cuantitativa continua", "NTD"],
    ["PAY_AMT1 a PAY_AMT6", "Monto pagado de septiembre a abril de 2005", "Cuantitativa continua", "NTD"],
    ["default", "Incumplimiento en el mes siguiente (octubre de 2005)", "Cualitativa nominal (binaria)", "1 = incumple, 0 = cumple"],
    ["BILL_AMT_PROM", "Promedio de BILL_AMT1 a BILL_AMT6", "Cuantitativa continua (derivada)", "NTD"],
    ["PAY_AMT_PROM", "Promedio de PAY_AMT1 a PAY_AMT6", "Cuantitativa continua (derivada)", "NTD"],
    ["GRUPO_EDAD", "Edad agrupada en 21-30, 31-40, 41-50, 51-60 y 61+", "Cualitativa ordinal (derivada)", "Grupos"],
], columns=["Variable", "Descripción", "Tipo", "Valores / unidad"])

descripcion

,Variable,Descripción,Tipo,Valores / unidad
0,ID,Identificador del cliente,Identificador,Entero
1,LIMIT_BAL,"Monto del crédito otorgado (NTD), incluye créd...",Cuantitativa continua,NTD
2,SEX,Sexo del cliente,Cualitativa nominal (binaria),"1 = hombre, 2 = mujer"
3,EDUCATION,Nivel de escolaridad,Cualitativa ordinal,"1 = posgrado, 2 = universidad, 3 = bachillerat..."
4,MARRIAGE,Estado civil,Cualitativa nominal,"1 = casado, 2 = soltero, 3 = otros"
5,AGE,Edad del cliente,Cuantitativa discreta,Años
6,PAY_0,Estado del pago de septiembre de 2005 (el más ...,Cualitativa ordinal,"-2 = sin consumo, -1 = pago total, 0 = pago mí..."
7,PAY_2 a PAY_6,"Estado del pago de agosto, julio, junio, mayo ...",Cualitativa ordinal,Misma escala que PAY_0
8,BILL_AMT1 a BILL_AMT6,Monto de la factura de septiembre a abril de 2005,Cuantitativa continua,NTD
9,PAY_AMT1 a PAY_AMT6,Monto pagado de septiembre a abril de 2005,Cuantitativa continua,NTD


**6. Variables auxiliares**

In [ ]:
import numpy as np
from scipy import stats

# Etiquetas legibles
df["SEXO"] = df["SEX"].map({1: "Hombre", 2: "Mujer"})
df["ESCOLARIDAD"] = df["EDUCATION"].map({1: "Posgrado", 2: "Universidad", 3: "Bachillerato", 4: "Otros"})
df["ESTADO_CIVIL"] = df["MARRIAGE"].map({1: "Casado", 2: "Soltero", 3: "Otros"})

# Grupos de edad
df["GRUPO_EDAD"] = pd.cut(df["AGE"], bins=[20, 30, 40, 50, 60, 100],
                          labels=["21-30", "31-40", "41-50", "51-60", "61+"])

# PAY_0 como categoría (texto) para poder agrupar
df["PAY_0_CAT"] = df["PAY_0"].astype(str)

# Promedios para evitar multicolinealidad
bill = [f"BILL_AMT{i}" for i in range(1, 7)]
pay_amt = [f"PAY_AMT{i}" for i in range(1, 7)]
df["BILL_AMT_PROM"] = df[bill].mean(axis=1)
df["PAY_AMT_PROM"] = df[pay_amt].mean(axis=1)

df[["SEXO", "ESCOLARIDAD", "ESTADO_CIVIL", "GRUPO_EDAD", "PAY_0_CAT"]].head()

,SEXO,ESCOLARIDAD,ESTADO_CIVIL,GRUPO_EDAD,PAY_0_CAT
0,Mujer,Universidad,Casado,21-30,2
1,Mujer,Universidad,Soltero,21-30,-1
2,Mujer,Universidad,Soltero,31-40,0
3,Mujer,Universidad,Casado,31-40,0
4,Hombre,Universidad,Casado,51-60,-1


# Fase 2: Exploración

**1. Tabla descriptiva y tasa de incumplimiento por categoría**

In [ ]:
def tabla_tasa(var, orden=None):
    t = (df.groupby(var, observed=True)["default"]
           .agg(Clientes="count", Incumplen="sum", Tasa_incumplimiento="mean")
           .reset_index().rename(columns={var: "Categoría"}))
    t.insert(0, "Variable", var)
    t["% de clientes"] = t["Clientes"] / len(df) * 100
    t["Tasa_incumplimiento"] *= 100
    if orden:
        t["Categoría"] = pd.Categorical(t["Categoría"], orden, ordered=True)
        t = t.sort_values("Categoría")
    return t

pay_orden = [str(i) for i in sorted(df["PAY_0"].unique())]

tablas = [
    tabla_tasa("SEXO"),
    tabla_tasa("ESCOLARIDAD", ["Posgrado", "Universidad", "Bachillerato", "Otros"]),
    tabla_tasa("ESTADO_CIVIL", ["Casado", "Soltero", "Otros"]),
    tabla_tasa("GRUPO_EDAD"),
    tabla_tasa("PAY_0_CAT", pay_orden),
]

tabla_cat = pd.concat(tablas, ignore_index=True)
tabla_cat["Categoría"] = tabla_cat["Categoría"].astype(str)
tabla_cat["Variable"] = tabla_cat["Variable"].replace({"PAY_0_CAT": "PAY_0"})
tabla_cat = tabla_cat.round(2)

print(f"N = {len(df):,} | Tasa global de incumplimiento = {df['default'].mean()*100:.2f}%")
tabla_cat

N = 30,000 | Tasa global de incumplimiento = 22.12%


,Variable,Categoría,Clientes,Incumplen,Tasa_incumplimiento,% de clientes
0,SEXO,Hombre,11888,2873,24.17,39.63
1,SEXO,Mujer,18112,3763,20.78,60.37
2,ESCOLARIDAD,Posgrado,10585,2036,19.23,35.28
3,ESCOLARIDAD,Universidad,14030,3330,23.73,46.77
4,ESCOLARIDAD,Bachillerato,4917,1237,25.16,16.39
5,ESCOLARIDAD,Otros,468,33,7.05,1.56
6,ESTADO_CIVIL,Casado,13659,3206,23.47,45.53
7,ESTADO_CIVIL,Soltero,15964,3341,20.93,53.21
8,ESTADO_CIVIL,Otros,377,89,23.61,1.26
9,GRUPO_EDAD,21-30,11013,2471,22.44,36.71


**2. Gráficas**

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

AZUL, MORADO, AMARILLO= "#1f4e79", "#EB8DE7", "#CFD166"
tasa_global = df["default"].mean() * 100

In [ ]:
t = tabla_cat[tabla_cat["Variable"] == "PAY_0"]

fig1 = go.Figure(go.Bar(
    x=t["Categoría"], y=t["Tasa_incumplimiento"], marker_color=MORADO,
    text=t["Tasa_incumplimiento"].map("{:.1f}%".format), textposition="outside",
    customdata=t["Clientes"],
    hovertemplate="PAY_0=%{x}<br>Tasa=%{y:.1f}%<br>n=%{customdata:,}<extra></extra>"))
fig1.add_hline(y=tasa_global, line_dash="dash", line_color=AZUL,
               annotation_text=f"Tasa global {tasa_global:.1f}%")
fig1.update_layout(template="plotly_white",
    title="Figura 1. Tasa de incumplimiento según el estado del último pago (PAY_0)",
    xaxis_title="PAY_0 (0 = pago mínimo, k = k meses de atraso)",
    yaxis_title="Tasa de incumplimiento (%)", yaxis_range=[0, 105])
fig1.show()

In [ ]:
fig2 = make_subplots(rows=1, cols=3, shared_yaxes=True,
                     subplot_titles=("Sexo", "Escolaridad", "Estado civil"))

for j, v in enumerate(["SEXO", "ESCOLARIDAD", "ESTADO_CIVIL"], start=1):
    t = tabla_cat[tabla_cat["Variable"] == v]
    fig2.add_trace(go.Bar(
        x=t["Categoría"], y=t["Tasa_incumplimiento"], marker_color=MORADO,
        text=t["Tasa_incumplimiento"].map("{:.1f}%".format), textposition="outside",
        customdata=t["Clientes"], showlegend=False,
        hovertemplate="%{x}<br>Tasa=%{y:.1f}%<br>n=%{customdata:,}<extra></extra>"),
        row=1, col=j)

fig2.update_yaxes(title_text="Tasa de incumplimiento (%)", range=[0, 35], row=1, col=1)
fig2.update_layout(template="plotly_white",
    title="Figura 2. Tasa de incumplimiento por sexo, escolaridad y estado civil")
fig2.show()

In [ ]:
t = tabla_cat[tabla_cat["Variable"] == "GRUPO_EDAD"]

fig3 = make_subplots(specs=[[{"secondary_y": True}]])
fig3.add_trace(go.Bar(x=t["Categoría"], y=t["Clientes"], name="Clientes",
                      marker_color=MORADO), secondary_y=False)
fig3.add_trace(go.Scatter(x=t["Categoría"], y=t["Tasa_incumplimiento"], name="Tasa (%)",
                          mode="lines+markers+text", line=dict(color=AMARILLO, width=3),
                          text=t["Tasa_incumplimiento"].map("{:.1f}%".format),
                          textposition="top center"), secondary_y=True)
fig3.update_yaxes(title_text="Número de clientes", secondary_y=False)
fig3.update_yaxes(title_text="Tasa de incumplimiento (%)", range=[0, 40], secondary_y=True)
fig3.update_layout(template="plotly_white",
    title="Figura 3. Clientes y tasa de incumplimiento por grupo de edad",
    xaxis_title="Grupo de edad (años)")
fig3.show()

In [ ]:
df["Estado"] = df["default"].map({0: "Cumple", 1: "Incumple"})

fig4 = px.box(df, x="Estado", y="LIMIT_BAL", color="Estado",
              color_discrete_map={"Cumple": AZUL, "Incumple": MORADO}, points=False)
fig4.update_layout(template="plotly_white",
    title="Figura 4. Límite de crédito (LIMIT_BAL) según el estado de incumplimiento",
    yaxis_title="LIMIT_BAL (NTD)", xaxis_title="", showlegend=False)
fig4.show()

**3. Matriz de correlación**

In [ ]:
vars_corr = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE", "PAY_0",
             "BILL_AMT_PROM", "PAY_AMT_PROM", "default"]
corr = df[vars_corr].corr(method="pearson").round(2)

fig5 = px.imshow(corr, text_auto=".2f", zmin=-1, zmax=1,
                 color_continuous_scale="RdBu_r", aspect="auto")
fig5.update_layout(template="plotly_white",
    title="Figura 5. Matriz de correlación de Pearson (BILL_AMT1-6 reemplazadas por su promedio)")
fig5.show()

# Fase 3: Modelado

**Preparación de variables**

In [ ]:
import numpy as np
import re
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

# PAY_0 agrupada
def agrupa_pay(x):
    if x == -2: return "Sin consumo"
    if x == -1: return "Pago total"
    if x == 0:  return "Pago minimo"
    if x == 1:  return "Atraso 1 mes"
    if x == 2:  return "Atraso 2 meses"
    return "Atraso 3+ meses"

df["PAY_0_G"] = df["PAY_0"].apply(agrupa_pay)

# Montos en unidades de 10,000 NT$
df["LIMIT_BAL_10k"]    = df["LIMIT_BAL"] / 10000
df["BILL_PROM_10k"]    = df["BILL_AMT_PROM"] / 10000
df["PAY_AMT_PROM_10k"] = df["PAY_AMT_PROM"] / 10000

**1. División 80/20 estratificada**

In [ ]:
train, test = train_test_split(df, test_size=0.2, stratify=df["default"], random_state=42)

print(len(train), len(test))
print(train["default"].mean(), test["default"].mean())   # deben ser casi iguales (~0.22)

24000 6000
0.22120833333333334 0.22116666666666668


**2. Ajuste del modelo**

In [ ]:
formula = ("default ~ LIMIT_BAL_10k + AGE + BILL_PROM_10k + PAY_AMT_PROM_10k"
           " + C(SEXO, Treatment('Hombre'))"
           " + C(ESCOLARIDAD, Treatment('Universidad'))"
           " + C(ESTADO_CIVIL, Treatment('Casado'))"
           " + C(PAY_0_G, Treatment('Pago minimo'))")

modelo = smf.logit(formula, data=train).fit(disp=False)
print(modelo.summary())
print("Pseudo R2 (McFadden):", round(modelo.prsquared, 3))

                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                24000
Model:                          Logit   Df Residuals:                    23984
Method:                           MLE   Df Model:                           15
Date:                Fri, 30 Oct 2026   Pseudo R-squ.:                  0.1647
Time:                        01:40:08   Log-Likelihood:                -10593.
converged:                       True   LL-Null:                       -12682.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                               coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------------
Intercept                                                   -1.5352      0.097    -15.752      0.000      -1.726      

**3. Tabla de odds ratios, IC 95 % y p-values**

In [ ]:
def limpia(n):
    m = re.search(r"C\((\w+).*\)\[T\.(.*)\]", n)
    return f"{m.group(1)}: {m.group(2)}" if m else n

ci = modelo.conf_int()
or_tab = pd.DataFrame({
    "Variable": modelo.params.index.map(limpia),
    "OR": np.exp(modelo.params.values),
    "IC95_inf": np.exp(ci[0].values),
    "IC95_sup": np.exp(ci[1].values),
    "p_value": modelo.pvalues.values
})
or_tab = or_tab[or_tab["Variable"] != "Intercept"].round(4).reset_index(drop=True)
or_tab

,Variable,OR,IC95_inf,IC95_sup,p_value
0,SEXO: Mujer,0.8435,0.7867,0.9045,0.0000
1,ESCOLARIDAD: Bachillerato,0.8980,0.8139,0.9908,0.0320
2,ESCOLARIDAD: Otros,0.2917,0.1927,0.4415,0.0000
3,ESCOLARIDAD: Posgrado,0.9811,0.9052,1.0634,0.6420
4,ESTADO_CIVIL: Otros,0.9809,0.7314,1.3156,0.8976
5,ESTADO_CIVIL: Soltero,0.8705,0.8044,0.9420,0.0006
6,PAY_0_G: Atraso 1 mes,3.7246,3.3779,4.1070,0.0000
7,PAY_0_G: Atraso 2 meses,13.9865,12.5543,15.5821,0.0000
8,PAY_0_G: Atraso 3+ meses,14.0802,11.0829,17.8880,0.0000
9,PAY_0_G: Pago total,1.8271,1.6403,2.0351,0.0000


**4. VIF (multicolinealidad)**

In [ ]:
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor

_, X_ = dmatrices(formula, data=train, return_type="dataframe")
vif = pd.DataFrame({
    "Variable": X_.columns.map(limpia),
    "VIF": [variance_inflation_factor(X_.values, i) for i in range(X_.shape[1])]
})
vif = vif[vif["Variable"] != "Intercept"].round(2)
vif

,Variable,VIF
1,SEXO: Mujer,1.02
2,ESCOLARIDAD: Bachillerato,1.20
3,ESCOLARIDAD: Otros,1.02
4,ESCOLARIDAD: Posgrado,1.26
5,ESTADO_CIVIL: Otros,1.02
6,ESTADO_CIVIL: Soltero,1.33
7,PAY_0_G: Atraso 1 mes,1.17
8,PAY_0_G: Atraso 2 meses,1.09
9,PAY_0_G: Atraso 3+ meses,1.02
10,PAY_0_G: Pago total,1.43


*VIF menor a 5 es aceptable, entre 5 y 10 es moderado y mayor a 10 es problemático. El uso de los promedios de BILL_ANT y PAR_AMT resolvió la multicolinealidad.*

*Todas las variables salen <5*

**5. Evaluación con un umbral de 0.5**

In [ ]:
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, roc_curve)

p_train = modelo.predict(train)
p_test  = modelo.predict(test)
y_train, y_test = train["default"], test["default"]

def metricas(y, p, u):
    pred = (p >= u).astype(int)
    return {"Umbral": round(u, 2),
            "Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred),
            "F1": f1_score(y, pred)}

print(metricas(y_test, p_test, 0.5))
print("AUC:", roc_auc_score(y_test, p_test))

{'Umbral': 0.5, 'Accuracy': 0.82, 'Precision': 0.6988727858293076, 'Recall': 0.32705350414468726, 'F1': 0.4455852156057495}
AUC: 0.7424804521670532


*El accuracy es alto: 82%. Sin embargo, tiene un recall bajo: 32.70%, ya que el modelo dejó de pasar a varios de los que sí incumplen.*

**6. Umbrales distintos**

In [ ]:
umbrales = np.arange(0.10, 0.61, 0.01)
f1s = [f1_score(y_train, (p_train >= u).astype(int)) for u in umbrales]
u_opt = umbrales[int(np.argmax(f1s))]
print("Umbral óptimo (máx. F1 en entrenamiento):", round(u_opt, 2))

comparacion = pd.DataFrame([metricas(y_test, p_test, 0.5),
                            metricas(y_test, p_test, u_opt)]).round(3)
comparacion

Umbral óptimo (máx. F1 en entrenamiento): 0.25


,Umbral,Accuracy,Precision,Recall,F1
0,0.50,0.82,0.699,0.327,0.446
1,0.25,0.78,0.503,0.501,0.502


*El umbral óptimo debería quedar por debajo de 0.5. Al bajarlo, el recall sube (detectan más incumplidores) y la precision baja (hay más falsas alarmas). Esa tabla de comparación es el centro de su discusión: ¿qué cuesta más para un banco, no detectar un incumplimiento o rechazar a un buen cliente?*

**7. Gráficas en Plotly**

Figura 6. Odds ratios con intervalos de confianza:

In [ ]:
d = or_tab.sort_values("OR")
fig6 = go.Figure(go.Scatter(
    x=d["OR"], y=d["Variable"], mode="markers",
    marker=dict(color=MORADO, size=9),
    error_x=dict(type="data", symmetric=False,
                 array=d["IC95_sup"] - d["OR"], arrayminus=d["OR"] - d["IC95_inf"],
                 color=AZUL),
    hovertemplate="%{y}<br>OR=%{x:.2f}<extra></extra>"))
fig6.add_vline(x=1, line_dash="dash", line_color="gray")
fig6.update_xaxes(type="log", title="Odds ratio (escala logarítmica)")
fig6.update_layout(template="plotly_white", height=600,
    title="Figura 6. Odds ratios e intervalos de confianza al 95 %")
fig6.show()

*Odds ratio es una medida estadística que cuantifica la fuerza de asociación entre dos variables*

Figura 7. Curva ROC:

In [ ]:
fpr, tpr, _ = roc_curve(y_test, p_test)
auc = roc_auc_score(y_test, p_test)

fig7 = go.Figure()
fig7.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"Modelo (AUC = {auc:.3f})",
                          line=dict(color=MORADO, width=3)))
fig7.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Azar",
                          line=dict(color="gray", dash="dash")))
fig7.update_layout(template="plotly_white",
    title="Figura 7. Curva ROC en el conjunto de prueba",
    xaxis_title="Tasa de falsos positivos", yaxis_title="Tasa de verdaderos positivos")
fig7.show()

*La curva ROC responde una pregunta: ¿qué tan bien separa el modelo a los clientes que incumplen de los que no, sin importar qué umbral se elija?*

Figura 8. Matrices de confusión (umbral 0.5 vs. óptimo):

In [ ]:
fig8 = make_subplots(rows=1, cols=2, subplot_titles=("Umbral 0.50", f"Umbral {u_opt:.2f}"))
for j, u in enumerate([0.5, u_opt], start=1):
    cm = confusion_matrix(y_test, (p_test >= u).astype(int))
    fig8.add_trace(go.Heatmap(z=cm, text=cm, texttemplate="%{text}",
        x=["Pred: Cumple", "Pred: Incumple"], y=["Real: Cumple", "Real: Incumple"],
        colorscale="Purples", showscale=False), row=1, col=j)
fig8.update_yaxes(autorange="reversed", automargin=True)
fig8.update_layout(margin=dict(l=120))
#fig8.update_yaxes(autorange="reversed")
#fig8.update_layout(template="plotly_white",
fig8.update_layout(title="Figura 8. Matriz de confusión en el conjunto de prueba")
fig8.show()

Figura 9: precision, recall y F1 según el umbral, con líneas verticales en 0.5 y en el umbral óptimo:

In [ ]:
#umbrales = np.arange(0.05, 0.71, 0.01)
umbrales = np.arange(0.05, 0.81, 0.01)

prec = [precision_score(y_test, (p_test >= u).astype(int), zero_division=0) for u in umbrales]
rec  = [recall_score(y_test, (p_test >= u).astype(int)) for u in umbrales]
f1_  = [f1_score(y_test, (p_test >= u).astype(int)) for u in umbrales]

fig9 = go.Figure()
fig9.add_trace(go.Scatter(x=umbrales, y=prec, mode="lines", name="Precision",
                          line=dict(color=AZUL, width=3)))
fig9.add_trace(go.Scatter(x=umbrales, y=rec, mode="lines", name="Recall",
                          line=dict(color=MORADO, width=3)))
fig9.add_trace(go.Scatter(x=umbrales, y=f1_, mode="lines", name="F1",
                          line=dict(color=AMARILLO, width=3)))

fig9.add_vline(x=0.5, line_dash="dash", line_color="gray",
               annotation_text="Umbral 0.50", annotation_position="top left")
fig9.add_vline(x=u_opt, line_dash="dash", line_color="black",
               annotation_text=f"Umbral óptimo {u_opt:.2f}", annotation_position="top right")

fig9.update_layout(template="plotly_white",
    title="Figura 9. Precision, recall y F1 según el umbral de clasificación",
    xaxis_title="Umbral de clasificación", yaxis_title="Valor de la métrica",
    yaxis_range=[0, 1])
fig9.show()

*Yo ando pensando en dejar como el umbral más alto a 0.71, porque con 0.80 cae hasta cero y se ve sucio el gráfico*

